# **Análise de Dados - Shoply**

## 1. ENTENDIMENTO DOS DADOS

### 1.1 Configuração inicial

In [ ]:
# Bibliotecas principais
import pandas as pd              # Manipulação de dados
import numpy as np               # Operações numéricas
import matplotlib.pyplot as plt  # Visualizações básicas
import seaborn as sns            # Visualizações estatísticas
from datetime import datetime
from datetime import timedelta
from pathlib import Path

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import shap

In [ ]:
# Configurações de display
pd.set_option("display.max_columns", None)   # Mostrar todas as colunas
pd.set_option("display.float_format", "{:,.2f}".format)  # Format numérico
sns.set(style="whitegrid", palette="viridis")  # Estilo padrão de gráficos

### 1.2 Carregamento dos dados

In [ ]:
# Carregar cada base separadamente
orders_raw = pd.read_csv("datasets/orders_final.csv")

### 1.3 Exploração Inicial de Dados

In [ ]:
# Avaliar tamanho das bases
orders_raw.shape

In [ ]:
# Avaliar head da base
orders_raw.head()

In [ ]:
orders_raw.info()

## 2. TRATAMENTO DOS DADOS

In [ ]:
orders = orders_raw.copy()

#### Orders

Tipagem de dados

In [ ]:
orders.info()

In [ ]:
# Garantir que IDs sejam tratados como strings e não números
orders["order_id"] = orders["order_id"].astype(str)
orders["customer_id"] = orders["customer_id"].astype(str)

# Tipagens corretas
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["order_value"]= pd.to_numeric(orders["order_value"], errors="coerce")
orders["discount_value"]= pd.to_numeric(orders["discount_value"], errors="coerce")
orders["sku_count"]  = pd.to_numeric(orders["sku_count"], errors="coerce")
orders["delivered_at"] = pd.to_datetime(orders["delivered_at"], errors="coerce")
orders["estimated_delivery_date"] = pd.to_datetime(orders["estimated_delivery_date"], errors="coerce")

Duplicatas

In [ ]:
# Contar quantas duplicatas existem
dup_mask = orders["order_id"].duplicated(keep=False)
print("Qtd de linhas duplicadas em order_id:", dup_mask.sum())

In [ ]:
# Ver primeira amostra das duplicatas
dup_sample = orders.loc[dup_mask].sort_values("order_id").head(6)
print("\nAmostra das duplicatas:\n", dup_sample)

In [ ]:
# remove duplicata de ID mantendo a 1ª ocorrência
orders = orders[~orders["order_id"].duplicated(keep="first")].copy()

Arrumar valores numéricos

In [ ]:
orders.describe()

In [ ]:
# order_value
neg_mask = orders["order_value"] < 0
neg_rate = neg_mask.mean()
print(f"Taxa de order_value < 0: {neg_rate:.2%}")

In [ ]:
# Se < 5%, remover
removed_neg = int(neg_mask.sum())
orders = orders[~neg_mask].copy()

In [ ]:
# Definir ideal pra desconto
mask_neg   = orders["discount_value"] < 0

n_total = len(orders)
n_neg   = int(mask_neg.sum())
print(f"Registros totais: {n_total}")
print(f"Inválidos (<0): {n_neg} ({n_neg/n_total:.2%})")

In [ ]:
# Correção
orders.loc[orders["discount_value"] < 0, "discount_value"] = 0

In [ ]:
# sku_count inválido → regra conservadora: mínimo 1, inteiro
# tudo que não for número vira NaN já na tipagem; agora imputamos 1
orders.loc[orders["sku_count"].isna() | (orders["sku_count"] < 1), "sku_count"] = 1

# força inteiro
orders["sku_count"] = orders["sku_count"].round().astype(int)

Lógica delivery

In [ ]:
# delivered_at antes da compra
mask_before_order = orders["delivered_at"].notna() & (orders["delivered_at"] < orders["order_date"])
# delivered_at preenchido quando não deveria
mask_shouldnt_have = orders["delivered_at"].notna() & orders["order_status"].isin(["cancelled","refunded","processing"])
# delivered_at vazio quando devia tá preenchido
mask_missing_but_should = orders["delivered_at"].isna() & orders["order_status"].isin(["delivered","returned"])
# estimated_delivery antes da compra
mask_bad_est = (orders["estimated_delivery_date"].notna() & (orders["estimated_delivery_date"] < orders["order_date"]))

print("Resumo inicial de erros detectados:\n")
print("Delivered_at antes da compra:", mask_before_order.sum())
print("Delivered_at preenchido quando não deveria:", mask_shouldnt_have.sum())
print("Delivered_at vazio quando devia tá preenchido:", mask_missing_but_should.sum())
print("Estimated_delivery antes da compra:", mask_bad_est.sum())

In [ ]:
# 1) delivered_at antes de order_date → limpar
orders.loc[mask_before_order, "delivered_at"] = pd.NaT

In [ ]:
# 2) delivered_at presente em status sem entrega → limpar
orders.loc[mask_shouldnt_have, "delivered_at"] = pd.NaT

In [ ]:
# 3) Arrumar tempo estimado
# SLA simples por estado
sla_map = {
    "AC": 8,"AL": 8,"AM": 8,"AP": 8,"BA": 8,"CE": 8,"PA": 8,"PB": 8,"PE": 8,"PI": 8,"MA": 8,"RN": 8,"RO": 8,"RR": 8,"SE": 8,"TO": 8,
    "DF": 5,"GO": 5,"MT": 5,"MS": 5,"PR": 5,"SC": 5,"RS": 5,
    "SP": 3,"RJ": 3,"MG": 3,"ES": 3
}

orders["sla_days"] = orders["delivery_state"].map(sla_map).fillna(5)

orders.loc[mask_bad_est, "estimated_delivery_date"] = (
    orders.loc[mask_bad_est, "order_date"] +
    pd.to_timedelta(orders.loc[mask_bad_est, "sla_days"], unit="D")
)

In [ ]:
# 4) delivered_at ausente mas deveria ter → imputar
#     regra: usar estimated_delivery_date; se NaT, usar order_date

mask_missing_but_should = orders["delivered_at"].isna() & orders["order_status"].isin(["delivered","returned"])
fill_est = orders.loc[mask_missing_but_should, "estimated_delivery_date"]
fill_ord = orders.loc[mask_missing_but_should, "order_date"]

orders.loc[mask_missing_but_should, "delivered_at"] = fill_est.fillna(fill_ord)

Normalização de campos categóricos

In [ ]:
orders["payment_method"].unique()

In [ ]:
# Normalização discreta de campos categóricos para reduzir ruído inadvertido
if "payment_method" in orders.columns:
    orders["payment_method"] = orders["payment_method"].str.lower().str.strip()
if "delivery_state" in orders.columns:
    orders["delivery_state"] = orders["delivery_state"].str.upper().str.strip()
if "order_status" in orders.columns:
    orders["order_status"] = orders["order_status"].str.lower().str.strip()
if "order_category" in orders.columns:
    orders["order_category"] = orders["order_category"].str.lower().str.strip()

Nulos

In [ ]:
# Nulos críticos
crit_cols_orders = ["order_id","customer_id","order_date","order_value"]
print(orders[crit_cols_orders].isna().sum())

In [ ]:
# Aplicar dropna
orders = orders.dropna(subset=["order_value"]).copy()

Arrumar payment methods

In [ ]:
orders['payment_method'].unique()

In [ ]:
orders["payment_method"] = (
    orders["payment_method"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"": None, "nan": None, "null": None, "-": None, "(blank)": None})
    .replace({
        "creditcard": "credit_card",
        "credit_card": "credit_card",
        "pixx": "pix",
        "pix": "pix",
        "boleto_bancario": "boleto",
        "boleto": "boleto",
        "paypal": "paypal",
    })
)


Arrumar state

In [ ]:
orders['delivery_state'].unique()

In [ ]:
orders["delivery_state"] = (
    orders["delivery_state"]
    .astype(str)
    .str.strip()
    .replace({
        "??": "XX"
    })
)

Arrumar datas erradas

In [ ]:
orders[orders["order_date"] > "2025-11-30"]

In [ ]:
orders = orders[orders["order_date"] <= "2025-11-30"]

Arrumar campanhas

In [ ]:
orders['campaign_id'].nunique()

In [ ]:
orders["campaign_id"] = (
    orders["campaign_id"]
    .astype(str)
    .str.strip()
    .replace({"": None, "nan": None, "null": None, "-": None, "(blank)": None, "N/A": None})
)

In [ ]:
orders.to_excel("orders_clean.xlsx")

### Features de pedido

In [ ]:
orders.head()

In [ ]:
# primeira compra do cliente
orders["is_new_customer"] = (
    orders.groupby("customer_id")["order_date"].transform("min") == orders["order_date"]
).astype(int)

In [ ]:
# Valor bruto e métricas de entrega
orders["total_order_value"] = orders["order_value"] + orders["discount_value"]
orders["pct_discount"] = orders["discount_value"]/orders["total_order_value"]
orders["days_to_delivery"] = (orders["delivered_at"] - orders["order_date"]).dt.days
orders["delivery_delay_days"] = (orders["delivered_at"] - orders["estimated_delivery_date"]).dt.days
orders["is_late_delivery"] = (orders["delivery_delay_days"] > 0).astype("Int8")

In [ ]:
# Particionamento temporal
orders["order_year"]  = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month
orders["order_month_year"] = orders["order_date"].dt.to_period("M")
orders["order_week"]  = orders["order_date"].dt.isocalendar().week.astype("Int64")

In [ ]:
# Sazonalidade
orders["holiday_tag"] = ""
orders.loc[orders["order_date"].dt.month == 11, "holiday_tag"] = "black_friday"
orders.loc[orders["order_date"].dt.month == 12, "holiday_tag"] = "christmas"
orders.loc[orders["order_date"].dt.month == 5,  "holiday_tag"] = "mothers_day"
orders["holiday_tag"] = orders["holiday_tag"].replace("", "normal")

In [ ]:
# Calcular quartis do ticket líquido
q1, q2, q3 = orders["order_value"].quantile([0.25, 0.50, 0.75])

print(f"Quartis do net_order_value:\nQ1={q1:.2f} | Q2={q2:.2f} | Q3={q3:.2f}")

In [ ]:
# Classificar ticket em 4 faixas
orders["basket_size_flag"] = pd.cut(
    orders["order_value"],
    bins=[-1, q1, q2, q3, orders["order_value"].max()],
    labels=["Q1_low","Q2_mid_low","Q3_mid_high","Q4_high"]
)

# High-ticket definido como Q4
orders["high_ticket_flag"] = (orders["order_value"] >= q3).astype("Int8")

In [ ]:
# Histórico incremental por cliente
orders = orders.sort_values(["customer_id","order_date","order_id"])
orders["customer_lifetime_orders"] = orders.groupby("customer_id").cumcount() + 1 # Número acumulado de pedidos por cliente (1ª, 2ª, 3ª compra etc.)
orders["customer_lifetime_gmv"] = orders.groupby("customer_id")["order_value"].cumsum() # GMV acumulado até o pedido atual

In [ ]:
# Ticket médio até o pedido atual
orders["customer_avg_ticket_to_date"] = (
    orders["customer_lifetime_gmv"] / orders["customer_lifetime_orders"]
)

In [ ]:
# Dias desde a última compra antes da compra atual
orders["prev_order_date"] = orders.groupby("customer_id")["order_date"].shift(1) # Data do pedido anterior do mesmo cliente
orders["days_since_last_order_before_purchase"] = (
    orders["order_date"] - orders["prev_order_date"]
).dt.days

In [ ]:
# Region (via UF) — nível pedido
uf_to_region = {
    "SP":"Sudeste","RJ":"Sudeste","MG":"Sudeste","ES":"Sudeste",
    "PR":"Sul","SC":"Sul","RS":"Sul",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste",
    "BA":"Nordeste","PE":"Nordeste","CE":"Nordeste","RN":"Nordeste","PB":"Nordeste","MA":"Nordeste","AL":"Nordeste","SE":"Nordeste","PI":"Nordeste",
    "PA":"Norte","AM":"Norte","RO":"Norte","RR":"Norte","AC":"Norte","AP":"Norte","TO":"Norte"
}
orders["region"] = orders["delivery_state"].map(uf_to_region).fillna("Desconhecida")


In [ ]:
# Identificar "novos" vs "recorrentes"

# Um cliente é "novo" se essa order é a primeira dele na série
orders = orders.sort_values(["customer_id", "order_date"])
orders["is_new_customer_period"] = orders["prev_order_date"].isna()


In [ ]:
orders.head()

In [ ]:
orders.to_excel("orders_features.xlsx")

### Features de diagnóstico

In [ ]:
orders.head()

In [ ]:
# Base de métricas gerais por mês pra facilitar diagnóstico

monthly = (
    orders.groupby("order_month_year")
    .agg(
        total_gmv=("order_value", "sum"),
        total_orders=("order_id", "nunique"),
        active_customers=("customer_id", "nunique"),
    )
)

monthly["aov"] = monthly["total_gmv"] / monthly["total_orders"]
monthly["orders_per_customer"] = monthly["total_orders"] / monthly["active_customers"]

In [ ]:
# Base auxiliar de infos pdiferenciando novos clientes de antigos
seg = (
    orders.groupby(["order_month_year", "is_new_customer_period"])
    .agg(
        gmv=("order_value", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
    )
    .reset_index()
)

# Pivot para ficar legível
seg = seg.pivot(
    index="order_month_year",
    columns="is_new_customer_period",
    values=["gmv", "orders", "customers"]
)

seg.columns = [
    f"{metric}_{'new' if flag else 'returning'}"
    for metric, flag in seg.columns]

In [ ]:
seg.head()

In [ ]:
# Unir tudo

# garantir formatos iguais para o join
monthly.index = monthly.index.astype(str)
seg.index = seg.index.astype(str)

diagnostic = monthly.join(seg, how="left")

In [ ]:
diagnostic.head()

In [ ]:
# Métricas derivadas

diagnostic["gmv_share_new"] = diagnostic["gmv_new"] / diagnostic["total_gmv"]

diagnostic["share_new"] = diagnostic["orders_new"] / diagnostic["total_orders"]

diagnostic["aov_new"] = diagnostic["gmv_new"] / diagnostic["orders_new"]
diagnostic["aov_returning"] = diagnostic["gmv_returning"] / diagnostic["orders_returning"] #Average Order Value.

diagnostic["orders_per_customer_new"] = (
    diagnostic["orders_new"] / diagnostic["customers_new"]
)
diagnostic["orders_per_customer_returning"] = (
    diagnostic["orders_returning"] / diagnostic["customers_returning"]
)

diagnostic.head(6)

In [ ]:
diagnostic.columns

### Features de clientes

In [ ]:
orders.columns

In [ ]:
orders.head()

In [ ]:
# Construção da base customers a partir de orders
customers = (
    orders[["customer_id"]]
    .dropna()
    .drop_duplicates()
    .sort_values("customer_id")
    .reset_index(drop=True)
)

In [ ]:
customers.head()

In [ ]:
# Período de relacionamento

tmp = (
    orders.groupby("customer_id")["order_date"]
    .agg(first_order_date="min", last_order_date="max")
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

today = pd.Timestamp.today().normalize()  # data de hoje sem horário

customers["total_days_as_customer"] = (today - customers["first_order_date"]).dt.days

In [ ]:
orders.columns

In [ ]:
# Volume e valor

tmp = (
    orders.groupby("customer_id")
    .agg(
        total_orders=("order_id","count"),
        total_gmv=("order_value","sum"),
        avg_order_value=("order_value","mean"),
        median_order_value=("order_value","median"),
        total_discount=("discount_value","sum"),
        avg_pct_discount=("pct_discount","mean"),
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Mix e perfil de compra

tmp = (
    orders.groupby("customer_id")
    .agg(
        unique_categories=("order_category","nunique"),
        avg_sku_per_order=("sku_count","mean"),
        pct_high_ticket=("high_ticket_flag","mean"),  # proporção Q4
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Logística e experiência

tmp = (
    orders.groupby("customer_id")
    .agg(
        avg_days_to_delivery=("days_to_delivery","mean"),
        pct_deliveries_late=("is_late_delivery","mean"),
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Região mais frequente

tmp = (
    orders.groupby("customer_id")["region"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
    .reset_index(name="most_frequent_region")
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Recompra e relacionamento

tmp = (
    orders.groupby("customer_id")["days_since_last_order_before_purchase"]
    .agg(avg_days_between_orders="mean", max_days_between_orders="max")
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Método de pagamento mais usado

tmp = (
    orders.groupby("customer_id")["payment_method"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
    .reset_index(name="most_used_payment")
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Mais algumas temporais

latest_date = orders["order_date"].max()
customers["days_since_last_order"] = (today - customers["last_order_date"]).dt.days
customers["is_active_90d"] = (customers["days_since_last_order"] <= 90).astype("Int8")

In [ ]:
# Coorte de entrada
customers["first_order_date"] = pd.to_datetime(customers["first_order_date"], errors="coerce")
customers["cohort_month"] = customers["first_order_date"].dt.to_period("M").astype(str)

In [ ]:
customers.columns

In [ ]:
customers.to_excel("customers_features.xlsx")

## 3. CRIAÇÃO DE FEATURES

In [ ]:
orders.to_excel("orders_clean.xlsx")

### Features de pedido

In [ ]:
orders.head()

In [ ]:
# primeira compra do cliente
orders["is_new_customer"] = (
    orders.groupby("customer_id")["order_date"].transform("min") == orders["order_date"]
).astype(int)

In [ ]:
# Valor bruto e métricas de entrega
orders["total_order_value"] = orders["order_value"] + orders["discount_value"]
orders["pct_discount"] = orders["discount_value"]/orders["total_order_value"]
orders["days_to_delivery"] = (orders["delivered_at"] - orders["order_date"]).dt.days
orders["delivery_delay_days"] = (orders["delivered_at"] - orders["estimated_delivery_date"]).dt.days
orders["is_late_delivery"] = (orders["delivery_delay_days"] > 0).astype("Int8")

In [ ]:
# Particionamento temporal
orders["order_year"]  = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month
orders["order_month_year"] = orders["order_date"].dt.to_period("M")
orders["order_week"]  = orders["order_date"].dt.isocalendar().week.astype("Int64")

In [ ]:
# Sazonalidade
orders["holiday_tag"] = ""
orders.loc[orders["order_date"].dt.month == 11, "holiday_tag"] = "black_friday"
orders.loc[orders["order_date"].dt.month == 12, "holiday_tag"] = "christmas"
orders.loc[orders["order_date"].dt.month == 5,  "holiday_tag"] = "mothers_day"
orders["holiday_tag"] = orders["holiday_tag"].replace("", "normal")

In [ ]:
# Calcular quartis do ticket líquido
q1, q2, q3 = orders["order_value"].quantile([0.25, 0.50, 0.75])

print(f"Quartis do net_order_value:\nQ1={q1:.2f} | Q2={q2:.2f} | Q3={q3:.2f}")

In [ ]:
# Classificar ticket em 4 faixas
orders["basket_size_flag"] = pd.cut(
    orders["order_value"],
    bins=[-1, q1, q2, q3, orders["order_value"].max()],
    labels=["Q1_low","Q2_mid_low","Q3_mid_high","Q4_high"]
)

# High-ticket definido como Q4
orders["high_ticket_flag"] = (orders["order_value"] >= q3).astype("Int8")

In [ ]:
# Histórico incremental por cliente
orders = orders.sort_values(["customer_id","order_date","order_id"])
orders["customer_lifetime_orders"] = orders.groupby("customer_id").cumcount() + 1 # Número acumulado de pedidos por cliente (1ª, 2ª, 3ª compra etc.)
orders["customer_lifetime_gmv"] = orders.groupby("customer_id")["order_value"].cumsum() # GMV acumulado até o pedido atual

In [ ]:
# Ticket médio até o pedido atual
orders["customer_avg_ticket_to_date"] = (
    orders["customer_lifetime_gmv"] / orders["customer_lifetime_orders"]
)

In [ ]:
# Dias desde a última compra antes da compra atual
orders["prev_order_date"] = orders.groupby("customer_id")["order_date"].shift(1) # Data do pedido anterior do mesmo cliente
orders["days_since_last_order_before_purchase"] = (
    orders["order_date"] - orders["prev_order_date"]
).dt.days

In [ ]:
# Region (via UF) — nível pedido
uf_to_region = {
    "SP":"Sudeste","RJ":"Sudeste","MG":"Sudeste","ES":"Sudeste",
    "PR":"Sul","SC":"Sul","RS":"Sul",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste",
    "BA":"Nordeste","PE":"Nordeste","CE":"Nordeste","RN":"Nordeste","PB":"Nordeste","MA":"Nordeste","AL":"Nordeste","SE":"Nordeste","PI":"Nordeste",
    "PA":"Norte","AM":"Norte","RO":"Norte","RR":"Norte","AC":"Norte","AP":"Norte","TO":"Norte"
}
orders["region"] = orders["delivery_state"].map(uf_to_region).fillna("Desconhecida")


In [ ]:
# Identificar "novos" vs "recorrentes"

# Um cliente é "novo" se essa order é a primeira dele na série
orders = orders.sort_values(["customer_id", "order_date"])
orders["is_new_customer_period"] = orders["prev_order_date"].isna()


In [ ]:
orders.head()

In [ ]:
orders.to_excel("orders_features.xlsx")

### Features de diagnóstico

In [ ]:
orders.head()

In [ ]:
# Base de métricas gerais por mês pra facilitar diagnóstico

monthly = (
    orders.groupby("order_month_year")
    .agg(
        total_gmv=("order_value", "sum"),
        total_orders=("order_id", "nunique"),
        active_customers=("customer_id", "nunique"),
    )
)

monthly["aov"] = monthly["total_gmv"] / monthly["total_orders"]
monthly["orders_per_customer"] = monthly["total_orders"] / monthly["active_customers"]

In [ ]:
# Base auxiliar de infos pdiferenciando novos clientes de antigos
seg = (
    orders.groupby(["order_month_year", "is_new_customer_period"])
    .agg(
        gmv=("order_value", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
    )
    .reset_index()
)

# Pivot para ficar legível
seg = seg.pivot(
    index="order_month_year",
    columns="is_new_customer_period",
    values=["gmv", "orders", "customers"]
)

seg.columns = [
    f"{metric}_{'new' if flag else 'returning'}"
    for metric, flag in seg.columns]

In [ ]:
seg.head()

In [ ]:
# Unir tudo

# garantir formatos iguais para o join
monthly.index = monthly.index.astype(str)
seg.index = seg.index.astype(str)

diagnostic = monthly.join(seg, how="left")

In [ ]:
diagnostic.head()

In [ ]:
# Métricas derivadas

diagnostic["gmv_share_new"] = diagnostic["gmv_new"] / diagnostic["total_gmv"]

diagnostic["share_new"] = diagnostic["orders_new"] / diagnostic["total_orders"]

diagnostic["aov_new"] = diagnostic["gmv_new"] / diagnostic["orders_new"]
diagnostic["aov_returning"] = diagnostic["gmv_returning"] / diagnostic["orders_returning"] #Average Order Value.

diagnostic["orders_per_customer_new"] = (
    diagnostic["orders_new"] / diagnostic["customers_new"]
)
diagnostic["orders_per_customer_returning"] = (
    diagnostic["orders_returning"] / diagnostic["customers_returning"]
)

diagnostic.head(6)

In [ ]:
diagnostic.columns

### Features de clientes

In [ ]:
orders.columns

In [ ]:
orders.head()

In [ ]:
# Construção da base customers a partir de orders
customers = (
    orders[["customer_id"]]
    .dropna()
    .drop_duplicates()
    .sort_values("customer_id")
    .reset_index(drop=True)
)

In [ ]:
customers.head()

In [ ]:
# Período de relacionamento

tmp = (
    orders.groupby("customer_id")["order_date"]
    .agg(first_order_date="min", last_order_date="max")
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

today = pd.Timestamp.today().normalize()  # data de hoje sem horário

customers["total_days_as_customer"] = (today - customers["first_order_date"]).dt.days

In [ ]:
orders.columns

In [ ]:
# Volume e valor

tmp = (
    orders.groupby("customer_id")
    .agg(
        total_orders=("order_id","count"),
        total_gmv=("order_value","sum"),
        avg_order_value=("order_value","mean"),
        median_order_value=("order_value","median"),
        total_discount=("discount_value","sum"),
        avg_pct_discount=("pct_discount","mean"),
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Mix e perfil de compra

tmp = (
    orders.groupby("customer_id")
    .agg(
        unique_categories=("order_category","nunique"),
        avg_sku_per_order=("sku_count","mean"),
        pct_high_ticket=("high_ticket_flag","mean"),  # proporção Q4
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Logística e experiência

tmp = (
    orders.groupby("customer_id")
    .agg(
        avg_days_to_delivery=("days_to_delivery","mean"),
        pct_deliveries_late=("is_late_delivery","mean"),
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Região mais frequente

tmp = (
    orders.groupby("customer_id")["region"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
    .reset_index(name="most_frequent_region")
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Recompra e relacionamento

tmp = (
    orders.groupby("customer_id")["days_since_last_order_before_purchase"]
    .agg(avg_days_between_orders="mean", max_days_between_orders="max")
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Método de pagamento mais usado

tmp = (
    orders.groupby("customer_id")["payment_method"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
    .reset_index(name="most_used_payment")
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Mais algumas temporais

latest_date = orders["order_date"].max()
customers["days_since_last_order"] = (today - customers["last_order_date"]).dt.days
customers["is_active_90d"] = (customers["days_since_last_order"] <= 90).astype("Int8")

In [ ]:
# Coorte de entrada
customers["first_order_date"] = pd.to_datetime(customers["first_order_date"], errors="coerce")
customers["cohort_month"] = customers["first_order_date"].dt.to_period("M").astype(str)

In [ ]:
customers.columns

In [ ]:
customers.to_excel("customers_features.xlsx")

In [ ]:
orders.to_excel("orders_clean.xlsx")

### Features de pedido

In [ ]:
orders.head()

In [ ]:
# primeira compra do cliente
orders["is_new_customer"] = (
    orders.groupby("customer_id")["order_date"].transform("min") == orders["order_date"]
).astype(int)

In [ ]:
# Valor bruto e métricas de entrega
orders["total_order_value"] = orders["order_value"] + orders["discount_value"]
orders["pct_discount"] = orders["discount_value"]/orders["total_order_value"]
orders["days_to_delivery"] = (orders["delivered_at"] - orders["order_date"]).dt.days
orders["delivery_delay_days"] = (orders["delivered_at"] - orders["estimated_delivery_date"]).dt.days
orders["is_late_delivery"] = (orders["delivery_delay_days"] > 0).astype("Int8")

In [ ]:
# Particionamento temporal
orders["order_year"]  = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month
orders["order_month_year"] = orders["order_date"].dt.to_period("M")
orders["order_week"]  = orders["order_date"].dt.isocalendar().week.astype("Int64")

In [ ]:
# Sazonalidade
orders["holiday_tag"] = ""
orders.loc[orders["order_date"].dt.month == 11, "holiday_tag"] = "black_friday"
orders.loc[orders["order_date"].dt.month == 12, "holiday_tag"] = "christmas"
orders.loc[orders["order_date"].dt.month == 5,  "holiday_tag"] = "mothers_day"
orders["holiday_tag"] = orders["holiday_tag"].replace("", "normal")

In [ ]:
# Calcular quartis do ticket líquido
q1, q2, q3 = orders["order_value"].quantile([0.25, 0.50, 0.75])

print(f"Quartis do net_order_value:\nQ1={q1:.2f} | Q2={q2:.2f} | Q3={q3:.2f}")

In [ ]:
# Classificar ticket em 4 faixas
orders["basket_size_flag"] = pd.cut(
    orders["order_value"],
    bins=[-1, q1, q2, q3, orders["order_value"].max()],
    labels=["Q1_low","Q2_mid_low","Q3_mid_high","Q4_high"]
)

# High-ticket definido como Q4
orders["high_ticket_flag"] = (orders["order_value"] >= q3).astype("Int8")

In [ ]:
# Histórico incremental por cliente
orders = orders.sort_values(["customer_id","order_date","order_id"])
orders["customer_lifetime_orders"] = orders.groupby("customer_id").cumcount() + 1 # Número acumulado de pedidos por cliente (1ª, 2ª, 3ª compra etc.)
orders["customer_lifetime_gmv"] = orders.groupby("customer_id")["order_value"].cumsum() # GMV acumulado até o pedido atual

In [ ]:
# Ticket médio até o pedido atual
orders["customer_avg_ticket_to_date"] = (
    orders["customer_lifetime_gmv"] / orders["customer_lifetime_orders"]
)

In [ ]:
# Dias desde a última compra antes da compra atual
orders["prev_order_date"] = orders.groupby("customer_id")["order_date"].shift(1) # Data do pedido anterior do mesmo cliente
orders["days_since_last_order_before_purchase"] = (
    orders["order_date"] - orders["prev_order_date"]
).dt.days

In [ ]:
# Region (via UF) — nível pedido
uf_to_region = {
    "SP":"Sudeste","RJ":"Sudeste","MG":"Sudeste","ES":"Sudeste",
    "PR":"Sul","SC":"Sul","RS":"Sul",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste",
    "BA":"Nordeste","PE":"Nordeste","CE":"Nordeste","RN":"Nordeste","PB":"Nordeste","MA":"Nordeste","AL":"Nordeste","SE":"Nordeste","PI":"Nordeste",
    "PA":"Norte","AM":"Norte","RO":"Norte","RR":"Norte","AC":"Norte","AP":"Norte","TO":"Norte"
}
orders["region"] = orders["delivery_state"].map(uf_to_region).fillna("Desconhecida")


In [ ]:
# Identificar "novos" vs "recorrentes"

# Um cliente é "novo" se essa order é a primeira dele na série
orders = orders.sort_values(["customer_id", "order_date"])
orders["is_new_customer_period"] = orders["prev_order_date"].isna()


In [ ]:
orders.head()

In [ ]:
orders.to_excel("orders_features.xlsx")

### Features de diagnóstico

In [ ]:
orders.head()

In [ ]:
# Base de métricas gerais por mês pra facilitar diagnóstico

monthly = (
    orders.groupby("order_month_year")
    .agg(
        total_gmv=("order_value", "sum"),
        total_orders=("order_id", "nunique"),
        active_customers=("customer_id", "nunique"),
    )
)

monthly["aov"] = monthly["total_gmv"] / monthly["total_orders"]
monthly["orders_per_customer"] = monthly["total_orders"] / monthly["active_customers"]

In [ ]:
# Base auxiliar de infos pdiferenciando novos clientes de antigos
seg = (
    orders.groupby(["order_month_year", "is_new_customer_period"])
    .agg(
        gmv=("order_value", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
    )
    .reset_index()
)

# Pivot para ficar legível
seg = seg.pivot(
    index="order_month_year",
    columns="is_new_customer_period",
    values=["gmv", "orders", "customers"]
)

seg.columns = [
    f"{metric}_{'new' if flag else 'returning'}"
    for metric, flag in seg.columns]

In [ ]:
seg.head()

In [ ]:
# Unir tudo

# garantir formatos iguais para o join
monthly.index = monthly.index.astype(str)
seg.index = seg.index.astype(str)

diagnostic = monthly.join(seg, how="left")

In [ ]:
diagnostic.head()

In [ ]:
# Métricas derivadas

diagnostic["gmv_share_new"] = diagnostic["gmv_new"] / diagnostic["total_gmv"]

diagnostic["share_new"] = diagnostic["orders_new"] / diagnostic["total_orders"]

diagnostic["aov_new"] = diagnostic["gmv_new"] / diagnostic["orders_new"]
diagnostic["aov_returning"] = diagnostic["gmv_returning"] / diagnostic["orders_returning"] #Average Order Value.

diagnostic["orders_per_customer_new"] = (
    diagnostic["orders_new"] / diagnostic["customers_new"]
)
diagnostic["orders_per_customer_returning"] = (
    diagnostic["orders_returning"] / diagnostic["customers_returning"]
)

diagnostic.head(6)

In [ ]:
diagnostic.columns

### Features de clientes

In [ ]:
orders.columns

In [ ]:
orders.head()

In [ ]:
# Construção da base customers a partir de orders
customers = (
    orders[["customer_id"]]
    .dropna()
    .drop_duplicates()
    .sort_values("customer_id")
    .reset_index(drop=True)
)

In [ ]:
customers.head()

In [ ]:
# Período de relacionamento

tmp = (
    orders.groupby("customer_id")["order_date"]
    .agg(first_order_date="min", last_order_date="max")
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

today = pd.Timestamp.today().normalize()  # data de hoje sem horário

customers["total_days_as_customer"] = (today - customers["first_order_date"]).dt.days

In [ ]:
orders.columns

In [ ]:
# Volume e valor

tmp = (
    orders.groupby("customer_id")
    .agg(
        total_orders=("order_id","count"),
        total_gmv=("order_value","sum"),
        avg_order_value=("order_value","mean"),
        median_order_value=("order_value","median"),
        total_discount=("discount_value","sum"),
        avg_pct_discount=("pct_discount","mean"),
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Mix e perfil de compra

tmp = (
    orders.groupby("customer_id")
    .agg(
        unique_categories=("order_category","nunique"),
        avg_sku_per_order=("sku_count","mean"),
        pct_high_ticket=("high_ticket_flag","mean"),  # proporção Q4
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Logística e experiência

tmp = (
    orders.groupby("customer_id")
    .agg(
        avg_days_to_delivery=("days_to_delivery","mean"),
        pct_deliveries_late=("is_late_delivery","mean"),
    )
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Região mais frequente

tmp = (
    orders.groupby("customer_id")["region"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
    .reset_index(name="most_frequent_region")
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Recompra e relacionamento

tmp = (
    orders.groupby("customer_id")["days_since_last_order_before_purchase"]
    .agg(avg_days_between_orders="mean", max_days_between_orders="max")
    .reset_index()
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Método de pagamento mais usado

tmp = (
    orders.groupby("customer_id")["payment_method"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
    .reset_index(name="most_used_payment")
)
customers = customers.merge(tmp, on="customer_id", how="left")

In [ ]:
# Mais algumas temporais

latest_date = orders["order_date"].max()
customers["days_since_last_order"] = (today - customers["last_order_date"]).dt.days
customers["is_active_90d"] = (customers["days_since_last_order"] <= 90).astype("Int8")

In [ ]:
# Coorte de entrada
customers["first_order_date"] = pd.to_datetime(customers["first_order_date"], errors="coerce")
customers["cohort_month"] = customers["first_order_date"].dt.to_period("M").astype(str)

In [ ]:
customers.columns

In [ ]:
customers.to_excel("customers_features.xlsx")

## 4. EDA

In [ ]:
# Estatísticas descritivas-chave

num_cols = [
    "total_orders","total_gmv","avg_order_value","median_order_value","total_discount",
    "unique_categories","avg_sku_per_order","pct_high_ticket",
    "avg_days_to_delivery","pct_deliveries_late",
    "avg_days_between_orders","max_days_between_orders","days_since_last_order","avg_pct_discount"
]
customers[num_cols].describe(percentiles=[0.25,0.5,0.75,0.9,0.95])

In [ ]:
# Distribuições rápidas (histogramas)
cols_hist = ["total_orders","total_gmv","avg_order_value","pct_high_ticket","pct_deliveries_late","days_since_last_order"]
for c in cols_hist:
    series = customers[c].dropna()
    plt.figure(figsize=(6,4))
    plt.hist(series, bins=40)
    plt.title(f"Distribuição: {c}")
    plt.xlabel(c); plt.ylabel("freq")
    plt.show()


In [ ]:
# =========================================================
# Coortes e atividade
# =========================================================
# Coorte mensal: quantos clientes entraram por mês
cohort_counts = customers.groupby("cohort_month")["customer_id"].nunique().reset_index(name="new_customers")
cohort_counts.head()

# Ativos 90d por coorte
cohort_active = (
    customers.groupby("cohort_month")["is_active_90d"]
    .mean().reset_index(name="active_90d_rate")
)
cohort_active.head()

# Mesclar e visualizar tendência
cohort_view = cohort_counts.merge(cohort_active, on="cohort_month", how="left").sort_values("cohort_month")
cohort_view.head(24)


In [ ]:
# =============================
# Plot geral do GMV
# =============================
plt.style.use("default")

cohort_view["active_90d_rate"].plot(kind="line", figsize=(10,4), title="active_90d_rate")
plt.show()

In [ ]:
# Top regiões
reg = customers["most_frequent_region"].value_counts(dropna=False).rename_axis("region").reset_index(name="customers")
reg

In [ ]:
# Método de pagamento dominante
pay = customers["most_used_payment"].value_counts(dropna=False).rename_axis("payment").reset_index(name="customers")
pay


RFM é um método comum pra segmentar clientes com base no comportamento de compra, para isso dividimos os clientes em e bins pra cada categoria (Reência, Frequência e Financeiro)


In [ ]:
# =========================================================
# Segmentação simples por valor e recência (RFM-lite)
# =========================================================
# R: recency = days_since_last_order (quanto menor, melhor)
# F: frequency = total_orders
# M: monetary = total_gmv
# Bins em tercis para simplicidade
customers["R_bin"] = pd.qcut(customers["days_since_last_order"], 3, labels=[3,2,1])  # 3=recente, 1=antigo
customers["F_bin"] = pd.qcut(customers["total_orders"].rank(method="first"), 3, labels=[1,2,3])  # 3=frequente
customers["M_bin"] = pd.qcut(customers["total_gmv"].rank(method="first"), 3, labels=[1,2,3])     # 3=alto valor

customers["RFM_score"] = customers[["R_bin","F_bin","M_bin"]].astype(int).sum(axis=1)

# Faixas de segmento
def label_segment(row):
    if row["RFM_score"] >= 8:   return "Champions"
    if row["RFM_score"] >= 6:   return "Leais"
    if row["RFM_score"] >= 4:   return "Potenciais"
    return "Em risco"

customers["segment_rfm"] = customers.apply(label_segment, axis=1)

In [ ]:
customers[["customer_id","R_bin","F_bin","M_bin","RFM_score","segment_rfm"]].tail(10)

In [ ]:
# Método de pagamento dominante
pay = customers["segment_rfm"].value_counts(dropna=False).rename_axis("segment_rfm").reset_index(name="customers")
pay

In [ ]:
# =========================================================
# Indicadores operacionais
# =========================================================
# Atraso médio por região
op_region = (
    customers
    .groupby("most_frequent_region")[["avg_days_to_delivery","pct_deliveries_late"]]
    .mean()
    .sort_values("pct_deliveries_late", ascending=False)
)
op_region


In [ ]:
# =========================================================
# Tabela executiva (KPIs resumidos)
# =========================================================
kpis = pd.DataFrame({
    "n_customers":[customers.shape[0]],
    "active_90d_rate":[customers["is_active_90d"].mean()],
    "avg_total_orders":[customers["total_orders"].mean()],
    "avg_gmv":[customers["total_gmv"].mean()],
    "avg_aov":[customers["avg_order_value"].mean()],
    "avg_days_between_orders":[customers["avg_days_between_orders"].mean()],
    "late_delivery_rate":[customers["pct_deliveries_late"].mean()],
})
kpis.T


In [ ]:
customers.to_csv("customers.csv", index=False, encoding="utf-8")

In [ ]:
# =============================
# Plot geral do GMV
# =============================
plt.style.use("default")

diagnostic["total_gmv"].plot(kind="line", figsize=(10,4), title="GMV total")
plt.show()

In [ ]:
# =============================
# 1. Drivers macro do GMV
# =============================
fig, axes = plt.subplots(2, 2, figsize=(12,8))

diagnostic["active_customers"].plot(ax=axes[0,0], title="Clientes Ativos")
diagnostic["total_orders"].plot(ax=axes[0,1], title="Total de Pedidos")
diagnostic["aov"].plot(ax=axes[1,0], title="Ticket Médio (AOV)")
diagnostic["orders_per_customer"].plot(ax=axes[1,1], title="Pedidos por Cliente")

plt.tight_layout()
plt.show()


In [ ]:
# =============================
# Quebra Novos vs Recorrentes
# =============================

fig, axes = plt.subplots(2, 2, figsize=(12,8))

diagnostic[["customers_new","customers_returning"]].plot(ax=axes[0,0], title="Clientes Novos vs Recorrentes")
diagnostic[["orders_new","orders_returning"]].plot(ax=axes[0,1], title="Pedidos: Novos vs Recorrentes")
diagnostic[["gmv_new","gmv_returning"]].plot(ax=axes[1,0], title="GMV: Novos vs Recorrentes")
diagnostic["gmv_share_new"].plot(ax=axes[1,1], title="% GMV vindo de Novos")

plt.tight_layout()
plt.show()

In [ ]:
# =============================
# Ticket médio por tipo de cliente
# =============================

fig, axes = plt.subplots(1, 2, figsize=(12,4))

diagnostic["aov_new"].plot(ax=axes[0], title="AOV - Novos")
diagnostic["aov_returning"].plot(ax=axes[1], title="AOV - Recorrentes")

plt.tight_layout()
plt.show()

In [ ]:
# =============================
# Frequência por tipo de cliente
# =============================

fig, axes = plt.subplots(1, 2, figsize=(12,4))

diagnostic["orders_per_customer_new"].plot(ax=axes[0], title="Pedidos por Cliente - Novos")
diagnostic["orders_per_customer_returning"].plot(ax=axes[1], title="Pedidos por Cliente - Recorrentes")

plt.tight_layout()
plt.show()

In [ ]:
# =============================
# Correlação para descobrir o driver
# =============================

corr = diagnostic.corr()
plt.figure(figsize=(10,8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.title("Correlação entre variáveis")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.tight_layout()
plt.show()

In [ ]:
# Calcular CAGR
# Garantir datetime no índice
diagnostic.index = pd.to_datetime(diagnostic.index.astype(str))

# Cortar tudo antes de 2020
diagnostic = diagnostic[diagnostic.index >= "2020-01-01"]

# Separar dois períodos
before = diagnostic[diagnostic.index < "2023-01-01"]
after  = diagnostic[diagnostic.index >= "2023-01-01"]

def calc_cagr(df, col):
    # precisa de pelo menos 3 linhas para pular a primeira com NaN
    if len(df) < 3:
        return np.nan

    # pula o primeiro mês (que pode ter NaN de retornos)
    start = df[col].iloc[1]
    end   = df[col].iloc[-1]

    years = (df.index[-1] - df.index[1]).days / 365.25

    if start <= 0 or years <= 0:
        return np.nan

    return (end / start) ** (1 / years) - 1

metrics = diagnostic.columns
results = []

for col in metrics:
    cagr_before = calc_cagr(before, col)
    cagr_after  = calc_cagr(after, col)
    results.append([col, cagr_before, cagr_after])

cagr_df = pd.DataFrame(results, columns=["metric", "CAGR_before_2023", "CAGR_after_2023"])
cagr_df



1) Crescimento geral (GMV, pedidos, clientes)

**Antes de 2023:**

* GMV: +64% ao ano

* Pedidos: +48%

* Clientes ativos: +48%
* AOV: + 11%

**Depois de 2023:**

* GMV: -1% ao ano (queda)

* Pedidos: +8%

* Clientes ativos: +8%
* **AOV: - 9% (queda)**

---
2) Ticket Médio — o principal problema

AOV geral:
* +11% → –9%

AOV por tipo:

* AOV_new: +12% → –16%

* AOV_returning: +9% → +1%


O ticket dos novos despencou brutalmente. O ticket dos recorrentes ficou quase flat (queda leve). **Esse é o maior driver isolado do GMV cair.**

---
3) Retenção (os recorrentes)

**Antes:**

* GMV_returning: +74%

* orders_returning: +60%

* customers_returning: +60%

**Depois:**

* GMV_returning: +5%

* orders_returning: +5%

* customers_returning: +5%

Antes, recorrentes eram a maior fonte de tração. Depois, eles só rondam o zero.
Esse é o segundo maior driver da estagnação.

---

4) Novos clientes (aquisição)

**Antes:**

* GMV_new: +55%

* orders_new: +39%

* customers_new: +39%

**Depois:**

* GMV_new: –6%

* orders_new: +11%

* customers_new: +11%

A aquisição ainda cresce (11% é decente). Mas o GMV dos novos cai por conta do ticket.

---

5) Mix de novos no GMV

* gmv_share_new: –6% → –5%

* share_new: –6% → +3%

O mix não é o problema principal. Ele fica estável, mas indica que temos menos recorrente (já identificado).

---

6) Frequência (orders per customer)

* 0% → 0%

A frequência ficou baixa e constante. Nem melhorou, nem piorou.

---

**🎯 Conclusão final por ordem de impacto no GMV pós-2023:**

* Ticket médio despencou (–9%), puxado principalmente por novos (–16%).

* Retenção deixou de ser motor (+74% → +5%).

* Aquisição desacelerou fortemente (+39% → +11%).

* Frequência estagnada não ajuda.

H1 – “Mix de novos deteriorou” (novos vindo com menos ticket / categorias baratas)

Outra avaliação: “Categorias de alto ticket perderam share”


In [ ]:
#Decompor GMV por categoria pré vs pós-2023.


# criar quartil do ticket (1 = mais baixo, 4 = mais alto)
orders["ticket_quartile"] = pd.qcut(
    orders["order_value"],
    4,
    labels=[1, 2, 3, 4]
).astype(int)

orders["order_year_month"] = orders["order_date"].dt.to_period("M")

high_share_by_month = (
    orders
    .groupby("order_year_month")
    .agg(
        pct_high_tickets=("ticket_quartile", lambda s: (s == 4).mean())
    )
)

# opcional: em %
high_share_by_month["pct_high_tickets"] = high_share_by_month["pct_high_tickets"] * 100


In [ ]:
high_share_by_month.tail(24)

In [ ]:
high_share_by_month["pct_high_tickets"].plot(kind="line", title="% de tickets no top 25% por mês")


Outra avaliação: “Descontos aumentaram e comeram ticket/margem”

In [ ]:
orders["discount_pct"] = orders["discount_value"] / orders["total_order_value"]

discount_summary = (
    orders
    .assign(period=lambda df: np.where(df["order_date"] < "2023-01-01", "pre_2023", "post_2023"))
    .groupby("period")
    .agg(
        avg_discount_pct=("discount_pct", "mean"),
        median_discount_pct=("discount_pct", "median"),
        aov=("order_value", "mean")
    )
)

print(discount_summary)
